# Phase 4b - Fake review detection

There is **no list of known fake reviews**, so we cannot train a normal classifier.
Instead we combine **warning signs** (rule-based signals) with **unsupervised** clustering of
reviewers, give every review a **suspicion score**, and then check a sample by hand.
Flagged reviews are called **"suspicious"**, never "proven fake" (as the synopsis says).

| # | Signal | Level | Idea |
|---|---|---|---|
| 1 | Near-duplicate text on different products | review | copied / template reviews (MinHash + LSH banding) |
| 2 | Stars vs text mismatch | review | 5 stars but the text reads negative (from Phase 4a) |
| 3 | Rating deviation | review | 3+ stars away from the product's average |
| 4 | Burst | review | most of a product's reviews arrive in one week, all extreme |
| 5 | Short generic extreme review | review | "great!!" with 1 or 5 stars |
| 6 | Unverified purchase | review | weak signal - Phase 3 showed unverified reviews are often detailed |
| 7 | One-sided prolific reviewer | reviewer | 5+ reviews, all 5-star or all 1-star |
| 8 | Outlier reviewer (KMeans) | reviewer | behaviour far from every normal group of reviewers |

Then: suspicion score -> flagged reviews -> **how product ratings change without them** ->
a **blind sample of 300 flagged + 300 normal reviews** for manual checking.

Run **top to bottom**. Expect about **15-25 minutes** on free Colab.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/amazon_reviews'
SAMPLE = '1pct'
LOCAL = '/content/parquet'
FIG_DIR = f'{PROJECT}/results/figures'
OUT_DIR = f'{PROJECT}/results/fake_reviews'
SEED = 42

In [ ]:
import os
JAVA = '/usr/lib/jvm/java-17-openjdk-amd64'
if not os.path.exists(JAVA):                      # skip the slow install if Java 17 is already there
    !apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
os.environ['JAVA_HOME'] = JAVA
!pip -q install pyspark==3.5.3

In [ ]:
!mkdir -p {LOCAL} {FIG_DIR} {OUT_DIR}
!cp -r {PROJECT}/parquet/reviews_clean_{SAMPLE} {PROJECT}/parquet/products_clean_{SAMPLE} {PROJECT}/parquet/sentiment_scores_{SAMPLE} {LOCAL}/

from pyspark.sql import SparkSession, Window, functions as F

spark = (SparkSession.builder
         .appName('Phase4b-FakeReviews')
         .master('local[*]')
         .config('spark.driver.memory', '9g')
         .config('spark.sql.shuffle.partitions', '16')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')

reviews = spark.read.parquet(f'{LOCAL}/reviews_clean_{SAMPLE}')
products = spark.read.parquet(f'{LOCAL}/products_clean_{SAMPLE}')
sentiment = spark.read.parquet(f'{LOCAL}/sentiment_scores_{SAMPLE}')

base = (reviews
        .select('review_id', 'user_id', 'parent_asin', 'rating', 'text', 'text_clean', 'tokens',
                'word_count', 'verified_purchase', 'timestamp', 'review_date')
        .join(products.select('parent_asin', 'product_title', 'average_rating'), 'parent_asin', 'left')
        .join(sentiment.select('review_id', 'p_negative', 'p_positive'), 'review_id', 'left')
        .withColumn('text_id', F.sha2('text_clean', 256))
        .cache())
print(f'reviews: {base.count():,}')

### Helpers: chart style and evaluation maths

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

BLUE, ORANGE, RED, GREY = '#2a78d6', '#eb6834', '#e34948', '#9a9893'
TEXT, MUTED, GRID = '#0b0b0b', '#52514e', '#e6e5e0'

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200, 'font.size': 10,
    'axes.edgecolor': GRID, 'axes.labelcolor': MUTED,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'axes.grid.axis': 'y', 'axes.ymargin': 0.12,
    'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlecolor': TEXT, 'axes.titlelocation': 'left',
    'legend.frameon': False,
})

def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(f'{FIG_DIR}/{name}.png', bbox_inches='tight', facecolor='white')
    plt.show()

def cohen_kappa(a, b):
    '''Agreement between two annotators beyond chance (1 = perfect, 0 = chance level).'''
    a, b = pd.Series(a).astype(str).reset_index(drop=True), pd.Series(b).astype(str).reset_index(drop=True)
    labels = sorted(set(a) | set(b))
    observed = (a == b).mean()
    expected = sum((a == l).mean() * (b == l).mean() for l in labels)
    return 1.0 if expected == 1 else (observed - expected) / (1 - expected)

def check_precision(sheet):
    '''Share of reviews judged fake-looking, for flagged vs normal reviews.'''
    out = (sheet.assign(fake_looking=(sheet['final_label'] == 'fake').astype(int))
                .groupby('suspicious')['fake_looking'].agg(['count', 'sum', 'mean']))
    out.columns = ['checked', 'judged_fake', 'share_fake']
    out.index = out.index.map({True: 'flagged by system', False: 'not flagged (random)'})
    return out

## 1. Signal: near-duplicate text on different products (MinHash + LSH banding)

Comparing every review with every other review is impossible at scale (431k x 431k pairs).
**MinHash** turns each review's set of words into 12 small numbers (a fingerprint). Two reviews
that share many words get many equal numbers.

**LSH banding** (the fast part): the 12 numbers are cut into **4 bands of 3**. Two reviews become
*candidates* only if **all 3 numbers in at least one band** are equal. Similar reviews almost always
match in some band; different reviews almost never do - so Spark only compares a small set of
candidate pairs, then computes the exact similarity for those.

| True similarity of two texts | Chance they become candidates |
|---|---|
| 0.9 | 99.5% |
| 0.8 | 94% |
| 0.5 | 42% (then rejected by the exact check) |
| 0.3 | 10% (then rejected) |

- Only reviews with **10+ words** are compared - short texts like "works great" are similar by nature.
- A pair counts if **Jaccard similarity >= 0.8** (80%+ of the distinct words shared) **and** the texts
  are on **different products**. Exact copies on 2+ products are included too.
- Buckets with more than 200 texts are skipped: those are generic phrases, and comparing them all
  is what makes the plain `approxSimilarityJoin` so slow.

In [ ]:
from pyspark.ml.feature import HashingTF, MinHashLSH

LSH_MIN_WORDS = 10      # only texts with 10+ words
MIN_JACCARD = 0.8       # 80%+ of distinct words shared
BANDS, ROWS = 4, 3      # 12 MinHash numbers -> 4 bands of 3
MAX_BUCKET = 200        # skip giant buckets of generic texts

t0 = time.time()
texts = (base.filter(F.col('word_count') >= LSH_MIN_WORDS)
         .groupBy('text_id')
         .agg(F.first('tokens').alias('tokens'), F.collect_set('parent_asin').alias('asins'))
         .withColumn('uniq', F.array_distinct('tokens')))
htf = HashingTF(inputCol='uniq', outputCol='vec', numFeatures=1 << 18, binary=True)
vecs = htf.transform(texts).select('text_id', 'asins', 'uniq', 'vec').cache()
print(f'distinct texts compared: {vecs.count():,}')

minhash = MinHashLSH(inputCol='vec', outputCol='hashes', numHashTables=BANDS * ROWS, seed=SEED).fit(vecs)

@F.udf('array<bigint>')
def flatten_hashes(hashes):
    return [int(h[0]) for h in hashes]

band_keys = F.array(*[
    F.concat_ws('_', F.lit(b), *[F.col('h')[b * ROWS + r].cast('string') for r in range(ROWS)])
    for b in range(BANDS)])

buckets = (minhash.transform(vecs)
           .withColumn('h', flatten_hashes('hashes'))
           .select('text_id', F.explode(band_keys).alias('band'))
           .withColumn('bucket_size', F.count('*').over(Window.partitionBy('band')))
           .cache())
skipped = buckets.filter(F.col('bucket_size') > MAX_BUCKET).select('band').distinct().count()
small = buckets.filter(F.col('bucket_size') <= MAX_BUCKET).select('text_id', 'band')
print(f'giant generic buckets skipped: {skipped}')

candidates = (small.alias('x').join(small.alias('y'), 'band')
              .filter(F.col('x.text_id') < F.col('y.text_id'))
              .select(F.col('x.text_id').alias('a'), F.col('y.text_id').alias('b'))
              .distinct())

info = vecs.select('text_id', 'asins', 'uniq')
pairs = (candidates
         .join(info.toDF('a', 'asins_a', 'uniq_a'), 'a')
         .join(info.toDF('b', 'asins_b', 'uniq_b'), 'b')
         .withColumn('jaccard', F.size(F.array_intersect('uniq_a', 'uniq_b')) /
                                F.size(F.array_union('uniq_a', 'uniq_b')))
         .filter(F.col('jaccard') >= MIN_JACCARD)
         .filter(F.size(F.array_union('asins_a', 'asins_b')) >= 2)
         .withColumn('jaccard_distance', F.round(1 - F.col('jaccard'), 3))
         .select('a', 'b', 'jaccard_distance')
         .cache())
print(f'near-duplicate pairs: {pairs.count():,}   ({time.time() - t0:.0f}s)')

exact_multi = vecs.filter(F.size('asins') >= 2).select('text_id')
dup_ids = (pairs.select(F.col('a').alias('text_id'))
           .union(pairs.select(F.col('b').alias('text_id')))
           .union(exact_multi).distinct()
           .withColumn('near_duplicate', F.lit(True)))
print(f'texts flagged as copied: {dup_ids.count():,}')

In [ ]:
# look at a few pairs to see what copied reviews look like
text_lookup = base.select('text_id', 'product_title', 'rating', 'text').dropDuplicates(['text_id'])
example_pairs = (pairs.orderBy('jaccard_distance').limit(8)
                 .join(text_lookup.toDF('a', 'product_a', 'rating_a', 'text_a'), 'a')
                 .join(text_lookup.toDF('b', 'product_b', 'rating_b', 'text_b'), 'b')
                 .select('jaccard_distance', 'product_a', 'text_a', 'product_b', 'text_b')
                 .toPandas())
pd.set_option('display.max_colwidth', 120)
example_pairs

## 2. Review-level signals

| Signal | Rule |
|---|---|
| Stars vs text mismatch | 4-5 stars and P(negative) >= 0.8, **or** 1-2 stars and P(positive) >= 0.8 |
| Rating deviation | rating is 3+ stars away from the product's average rating (Amazon's full average) |
| Burst | 3+ reviews within +/-3 days, those are **50%+ of the product's reviews**, and their average is >= 4.5 or <= 1.5 |
| Short generic extreme | 3 words or fewer (after stop words) and 1 or 5 stars |
| Unverified | Amazon did not confirm the purchase |

**Why the burst rule looks at the share of a product's reviews:** Phase 3 showed the biggest
single-day bursts were Echo Dot / Fire TV Stick right after Christmas - genuine gift reviews.
Popular products get many reviews every week, so a busy week is only suspicious when it is
**most** of the product's reviews.

In [ ]:
DAY = 86400
w_near = Window.partitionBy('parent_asin').orderBy('ts').rangeBetween(-3 * DAY, 3 * DAY)
w_product = Window.partitionBy('parent_asin')

signals = (base
    .join(dup_ids, 'text_id', 'left')
    .withColumn('ts', (F.col('timestamp') / 1000).cast('long'))
    .withColumn('week_reviews', F.count('*').over(w_near))
    .withColumn('week_avg_rating', F.avg('rating').over(w_near))
    .withColumn('product_reviews', F.count('*').over(w_product))
    .withColumn('near_duplicate', F.coalesce('near_duplicate', F.lit(False)))
    .withColumn('text_rating_mismatch',
                ((F.col('rating') >= 4) & (F.col('p_negative') >= 0.8)) |
                ((F.col('rating') <= 2) & (F.col('p_positive') >= 0.8)))
    .withColumn('rating_deviation', F.abs(F.col('rating') - F.col('average_rating')) >= 3)
    .withColumn('burst', (F.col('week_reviews') >= 3) &
                         (F.col('week_reviews') / F.col('product_reviews') >= 0.5) &
                         ((F.col('week_avg_rating') >= 4.5) | (F.col('week_avg_rating') <= 1.5)))
    .withColumn('short_generic', (F.col('word_count') <= 3) & F.col('rating').isin(1.0, 5.0))
    .withColumn('unverified', ~F.col('verified_purchase'))
    .na.fill(False, ['text_rating_mismatch', 'rating_deviation', 'burst', 'short_generic', 'unverified'])
    .cache())
print(f'reviews with signals: {signals.count():,}')

## 3. Reviewer-level signals: one-sided reviewers and KMeans outliers

We describe every reviewer with **3+ reviews** by 11 behaviour numbers (review counts on a
**log scale**, so the few reviewers with 100+ reviews do not dominate), scale them, and
group reviewers with **KMeans**. Most reviewers fall close to one of the group centres.
Reviewers **far from every centre** (top 1% distance) behave unusually -> *outlier reviewer*.

The number of groups *k* is chosen by the **silhouette score** (how well separated the groups are).

In [ ]:
user_feats = (signals.groupBy('user_id').agg(
        F.count('*').alias('n_reviews'),
        F.avg('rating').alias('avg_rating'),
        F.stddev_pop('rating').alias('rating_std'),
        F.avg((F.col('rating') == 5).cast('double')).alias('share_5_star'),
        F.avg((F.col('rating') == 1).cast('double')).alias('share_1_star'),
        F.avg(F.col('verified_purchase').cast('double')).alias('share_verified'),
        F.avg('word_count').alias('avg_words'),
        F.avg(F.abs(F.col('rating') - F.col('average_rating'))).alias('avg_deviation'),
        F.avg(F.col('near_duplicate').cast('double')).alias('share_duplicate'),
        F.avg(F.col('text_rating_mismatch').cast('double')).alias('share_mismatch'),
        F.countDistinct('review_date').alias('active_days'))
    .withColumn('reviews_per_active_day', F.col('n_reviews') / F.col('active_days'))
    # log scale: a few reviewers have 100+ reviews; without log they dominate the distances
    .withColumn('log_n_reviews', F.log1p('n_reviews'))
    .withColumn('log_reviews_per_day', F.log1p('reviews_per_active_day'))
    .filter(F.col('n_reviews') >= 3)
    .na.fill(0.0)
    .cache())

FEATURES = ['log_n_reviews', 'avg_rating', 'rating_std', 'share_5_star', 'share_1_star', 'share_verified',
            'avg_words', 'avg_deviation', 'share_duplicate', 'share_mismatch', 'log_reviews_per_day']
# readable (non-log) versions for the cluster profile table
PROFILE_COLS = ['n_reviews', 'avg_rating', 'rating_std', 'share_5_star', 'share_1_star', 'share_verified',
                'avg_words', 'avg_deviation', 'share_duplicate', 'share_mismatch', 'reviews_per_active_day']
print(f'reviewers with 3+ reviews: {user_feats.count():,}')

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

prep = Pipeline(stages=[VectorAssembler(inputCols=FEATURES, outputCol='raw'),
                        StandardScaler(inputCol='raw', outputCol='features', withMean=True, withStd=True)])
users_scaled = prep.fit(user_feats).transform(user_feats).cache()

sil_rows, km_models = [], {}
for k in range(3, 8):
    km = KMeans(k=k, seed=SEED, featuresCol='features').fit(users_scaled)
    sil = ClusteringEvaluator(featuresCol='features').evaluate(km.transform(users_scaled))
    sil_rows.append({'k': k, 'silhouette': round(sil, 4)}); km_models[k] = km
    print(f'k={k}  silhouette={sil:.4f}')
silhouette = pd.DataFrame(sil_rows)
silhouette.to_csv(f'{OUT_DIR}/kmeans_silhouette_{SAMPLE}.csv', index=False)
BEST_K = int(silhouette.loc[silhouette['silhouette'].idxmax(), 'k'])
print('chosen k =', BEST_K)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(silhouette['k'], silhouette['silhouette'], color=BLUE, linewidth=2, marker='o', markersize=6)
best = silhouette.loc[silhouette['k'] == BEST_K].iloc[0]
ax.annotate(f"chosen k = {BEST_K}", (best['k'], best['silhouette']), xytext=(0, 8),
            textcoords='offset points', ha='center', color=TEXT)
ax.set_xlabel('number of clusters (k)')
ax.set_ylabel('silhouette score')
ax.set_title('Choosing k for reviewer clustering')
save_fig(fig, '4b_kmeans_silhouette')

In [ ]:
km = km_models[BEST_K]
centers = [np.asarray(c) for c in km.clusterCenters()]

@F.udf('double')
def distance_to_centre(vec, cluster):
    return float(np.linalg.norm(vec.toArray() - centers[cluster]))

clustered = (km.transform(users_scaled)
             .withColumn('distance', distance_to_centre('features', 'prediction'))
             .withColumnRenamed('prediction', 'cluster')
             .cache())
OUTLIER_CUTOFF = clustered.approxQuantile('distance', [0.99], 0.001)[0]
print(f'outlier = distance above {OUTLIER_CUTOFF:.2f} (top 1%)')

# what does each group of reviewers look like? (average of the raw numbers)
profiles = (clustered.groupBy('cluster')
            .agg(F.count('*').alias('reviewers'), *[F.round(F.avg(c), 2).alias(c) for c in PROFILE_COLS])
            .orderBy('cluster').toPandas())
profiles.to_csv(f'{OUT_DIR}/kmeans_cluster_profiles_{SAMPLE}.csv', index=False)
profiles

**Read the cluster profiles above and give each group a name for the report**, for example
"happy casual buyers" (avg rating ~4.8, few reviews), "critical reviewers" (low avg rating, long texts),
"heavy reviewers" (many reviews per active day). Outliers are reviewers who fit none of them well.

In [ ]:
user_flags = (clustered
    .withColumn('outlier_reviewer', F.col('distance') > OUTLIER_CUTOFF)
    .withColumn('one_sided_reviewer', (F.col('n_reviews') >= 5) &
                ((F.col('share_5_star') == 1.0) | (F.col('share_1_star') == 1.0)))
    .select('user_id', 'cluster', 'distance', 'outlier_reviewer', 'one_sided_reviewer'))

signals = (signals.join(user_flags, 'user_id', 'left')
           .na.fill(False, ['outlier_reviewer', 'one_sided_reviewer'])
           .cache())
print(f"outlier reviewers: {user_flags.filter('outlier_reviewer').count():,}   "
      f"one-sided reviewers: {user_flags.filter('one_sided_reviewer').count():,}")

## 4. Suspicion score

Each signal adds points. Stronger evidence gets more points:

| Signal | Points | Why |
|---|---|---|
| near_duplicate | 3.0 | copying text across products is the clearest sign (Jindal & Liu 2008) |
| text_rating_mismatch | 2.0 | the words contradict the stars |
| rating_deviation | 1.5 | behavioural signal (Mukherjee et al. 2013) |
| burst | 1.5 | behavioural signal |
| outlier_reviewer | 1.5 | unusual reviewer overall |
| one_sided_reviewer | 1.0 | can also be an honest enthusiastic buyer |
| short_generic | 1.0 | common in honest reviews too |
| unverified | 0.5 | weak - Phase 3 showed unverified reviews are often detailed (Vine) |

A review is **suspicious** when its score is **3.0 or more** - i.e. one strong signal
(near-duplicate) or at least two others together. One weak signal alone never flags a review.

In [ ]:
WEIGHTS = {'near_duplicate': 3.0, 'text_rating_mismatch': 2.0, 'rating_deviation': 1.5, 'burst': 1.5,
           'outlier_reviewer': 1.5, 'one_sided_reviewer': 1.0, 'short_generic': 1.0, 'unverified': 0.5}
THRESHOLD = 3.0

score_expr = sum(F.col(s).cast('double') * w for s, w in WEIGHTS.items())
scored = (signals
          .withColumn('suspicion_score', score_expr)
          .withColumn('signals_count', sum(F.col(s).cast('int') for s in WEIGHTS))
          .withColumn('suspicious', F.col('suspicion_score') >= THRESHOLD)
          .cache())

n_total = scored.count()
n_flagged = scored.filter('suspicious').count()
print(f'suspicious reviews: {n_flagged:,} of {n_total:,}  ({100 * n_flagged / n_total:.2f}%)')

prevalence = pd.DataFrame([
    {'signal': s, 'points': w,
     'reviews': scored.filter(F.col(s)).count()} for s, w in WEIGHTS.items()])
prevalence['pct_of_all'] = (100 * prevalence['reviews'] / n_total).round(2)
prevalence['pct_of_flagged'] = [
    round(100 * scored.filter(F.col('suspicious') & F.col(s)).count() / max(n_flagged, 1), 1) for s in WEIGHTS]
prevalence.to_csv(f'{OUT_DIR}/signal_prevalence_{SAMPLE}.csv', index=False)
prevalence

In [ ]:
p = prevalence.iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4.2))
bars = ax.barh(p['signal'], p['pct_of_all'], height=0.6, color=BLUE, edgecolor='white', linewidth=1.5)
ax.bar_label(bars, labels=[f'{v:.2f}%  ({int(n):,})' for v, n in zip(p['pct_of_all'], p['reviews'])],
             padding=3, color=TEXT, fontsize=9)
ax.set_xlim(0, p['pct_of_all'].max() * 1.35)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=1))
ax.grid(axis='x', visible=True)
ax.grid(axis='y', visible=False)
ax.set_title('How often each warning sign appears (share of all reviews)')
save_fig(fig, '4b_signal_prevalence')

In [ ]:
buckets = (scored.withColumn('score_bucket',
               F.when(F.col('suspicion_score') == 0, '0')
                .when(F.col('suspicion_score') < 1.5, '0.5-1')
                .when(F.col('suspicion_score') < 3.0, '1.5-2.5')
                .when(F.col('suspicion_score') < 4.5, '3-4')
                .otherwise('4.5+'))
           .groupBy('score_bucket').count().toPandas())
buckets['pct'] = (100 * buckets['count'] / buckets['count'].sum()).round(2)
buckets.to_csv(f'{OUT_DIR}/score_distribution_{SAMPLE}.csv', index=False)
buckets

In [ ]:
ORDER = ['0', '0.5-1', '1.5-2.5', '3-4', '4.5+']
b = buckets.set_index('score_bucket').reindex(ORDER).fillna(0)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
bars = ax.bar(ORDER, b['pct'], width=0.6, edgecolor='white', linewidth=2,
              color=[RED if o in ('3-4', '4.5+') else GREY for o in ORDER])
ax.bar_label(bars, labels=[f'{v:.1f}%' for v in b['pct']], padding=3, color=TEXT)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_xlabel('suspicion score')
ax.set_ylabel('share of reviews')
ax.set_title(f'Suspicion scores (red = flagged, score >= {THRESHOLD:g})')
save_fig(fig, '4b_score_distribution')

## 5. How much do suspicious reviews change product ratings?

For every product with **10+ reviews in the sample**, compare the average rating of **all**
reviews with the average after **removing suspicious** ones.

In [ ]:
impact = (scored.groupBy('parent_asin', 'product_title')
          .agg(F.count('*').alias('reviews'),
               F.sum(F.col('suspicious').cast('int')).alias('flagged'),
               F.avg('rating').alias('avg_all'),
               F.avg(F.when(~F.col('suspicious'), F.col('rating'))).alias('avg_without_flagged'))
          .filter(F.col('reviews') >= 10)
          .withColumn('change', F.col('avg_without_flagged') - F.col('avg_all'))
          .toPandas())
impact.to_csv(f'{OUT_DIR}/product_rating_impact_{SAMPLE}.csv', index=False)

changed = impact[impact['change'].abs() >= 0.1]
print(f'products with 10+ reviews: {len(impact):,}')
print(f'rating changes by 0.1+ stars: {len(changed):,} ({100 * len(changed) / max(len(impact), 1):.1f}%)')
print(f"inflated (drops when flagged removed): {(impact['change'] <= -0.1).sum():,}   "
      f"deflated (rises): {(impact['change'] >= 0.1).sum():,}")
impact.reindex(impact['change'].abs().sort_values(ascending=False).index).head(12).round(2)

In [ ]:
top = impact.reindex(impact['change'].abs().sort_values(ascending=False).index).head(12).iloc[::-1]
names = [t[:45] + '...' if isinstance(t, str) and len(t) > 45 else str(t) for t in top['product_title']]
y = range(len(top))
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.hlines(y, top['avg_all'], top['avg_without_flagged'], color=GRID, linewidth=3)
ax.scatter(top['avg_all'], y, color=GREY, s=60, zorder=3, label='all reviews')
ax.scatter(top['avg_without_flagged'], y, color=BLUE, s=60, zorder=3, label='suspicious removed')
ax.set_yticks(list(y), names, fontsize=8)
ax.set_xlabel('average star rating')
ax.grid(axis='x', visible=True)
ax.grid(axis='y', visible=False)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncols=2)
ax.set_title('Products whose rating changes most without suspicious reviews')
save_fig(fig, '4b_rating_impact')

## 6. Most suspicious reviewers

In [ ]:
suspicious_reviewers = (scored.groupBy('user_id')
    .agg(F.count('*').alias('reviews'),
         F.sum(F.col('suspicious').cast('int')).alias('flagged_reviews'),
         F.round(F.avg('suspicion_score'), 2).alias('avg_score'),
         F.round(F.avg('rating'), 2).alias('avg_rating'),
         F.max(F.col('near_duplicate').cast('int')).alias('has_duplicates'),
         F.max(F.col('outlier_reviewer').cast('int')).alias('outlier'))
    .filter(F.col('flagged_reviews') >= 2)
    .orderBy(F.desc('flagged_reviews'), F.desc('avg_score'))
    .toPandas())
suspicious_reviewers.to_csv(f'{OUT_DIR}/suspicious_reviewers_{SAMPLE}.csv', index=False)
print(f'reviewers with 2+ suspicious reviews: {len(suspicious_reviewers):,}')
suspicious_reviewers.head(15)

## 7. Save the scores

In [ ]:
SIGNALS = list(WEIGHTS)
(scored.select('review_id', 'user_id', 'parent_asin', 'rating', *SIGNALS,
               'suspicion_score', 'signals_count', 'suspicious', 'cluster')
       .write.mode('overwrite').parquet(f'{LOCAL}/fake_scores_{SAMPLE}'))
!rm -rf {PROJECT}/parquet/fake_scores_{SAMPLE} && cp -r {LOCAL}/fake_scores_{SAMPLE} {PROJECT}/parquet/
print('saved:', spark.read.parquet(f'{PROJECT}/parquet/fake_scores_{SAMPLE}').count(), 'rows')

## 8. Manual check sheet (300 flagged + 300 normal, blind)

This is how the synopsis evaluates the detector: people read reviews and judge them,
**without knowing** which ones the system flagged (so they are not biased).

The sheet mixes 300 flagged and 300 random normal reviews in random order. The answer key
(which ones were flagged) is saved in a **separate** file - do not open it while labelling.

### Labelling guide (give this to both annotators)
For each review write one word in your column:
- **fake** - looks paid / copied / not a real experience: generic praise that could fit any
  product, mentions a different product, marketing language, copied text, nonsense
- **genuine** - describes a real experience with specific details (good or bad)
- **unsure** - cannot tell

Two people label **separately** (e.g. you and one friend). Then for each row agree on a
`final_label` (if one says fake and the other genuine, discuss it).

In [ ]:
cols = ['review_id', 'product_title', 'rating', 'verified_purchase', 'review_date', 'text', 'suspicious', 'suspicion_score']
flag_s = scored.filter('suspicious').orderBy(F.rand(SEED)).limit(300).select(*cols)
norm_s = scored.filter(~F.col('suspicious')).orderBy(F.rand(SEED)).limit(300).select(*cols)
both = flag_s.union(norm_s).toPandas().sample(frac=1, random_state=SEED).reset_index(drop=True)
both.insert(0, 'check_id', range(1, len(both) + 1))

sheet = both[['check_id', 'product_title', 'rating', 'verified_purchase', 'review_date', 'text']].copy()
sheet['annotator_1'] = ''
sheet['annotator_2'] = ''
sheet['final_label'] = ''
answer_key = both[['check_id', 'review_id', 'suspicious', 'suspicion_score']]

sheet.to_csv(f'{OUT_DIR}/manual_check_sheet.csv', index=False)
answer_key.to_csv(f'{OUT_DIR}/manual_check_ANSWER_KEY.csv', index=False)
print('saved manual_check_sheet.csv (to label) and manual_check_ANSWER_KEY.csv (do not open while labelling)')
sheet.head(3)

### After labelling: compute the results

1. Download `manual_check_sheet.csv` from Drive, open it in **Excel / Google Sheets**, fill
   `annotator_1`, `annotator_2` and `final_label` with **fake / genuine / unsure**.
2. Save it as `manual_check_labelled.csv` in the same Drive folder (`results/fake_reviews/`).
3. Run the cell below.

**How to read it:** if the detector works, the share judged "fake" is clearly **higher for flagged**
reviews than for the random normal ones - that difference is the evidence for the report.
**Cohen's kappa** shows whether the two annotators agree (above 0.6 = good agreement).

In [ ]:
labelled_path = f'{OUT_DIR}/manual_check_labelled.csv'
if not os.path.exists(labelled_path):
    print('Not labelled yet - fill the sheet first (see the steps above).')
else:
    labelled = pd.read_csv(labelled_path).merge(pd.read_csv(f'{OUT_DIR}/manual_check_ANSWER_KEY.csv'), on='check_id')
    for c in ['annotator_1', 'annotator_2', 'final_label']:
        labelled[c] = labelled[c].astype(str).str.strip().str.lower()
    print(f"Cohen's kappa between annotators: {cohen_kappa(labelled['annotator_1'], labelled['annotator_2']):.3f}")
    result = check_precision(labelled[labelled['final_label'].isin(['fake', 'genuine'])])
    result.to_csv(f'{OUT_DIR}/manual_check_result.csv')
    display(result)

## Summary - copy everything this cell prints and send it back

In [ ]:
print('=== 1. COPIED REVIEWS ===')
print(f'texts compared: {vecs.count():,}   near-duplicate pairs: {pairs.count():,}   texts flagged as copied: {dup_ids.count():,}')

print('\n=== 3. KMEANS ===')
print(silhouette.to_string(index=False)); print('chosen k =', BEST_K)
print(profiles.to_string(index=False))
print(f"outlier reviewers: {user_flags.filter('outlier_reviewer').count():,}   "
      f"one-sided reviewers: {user_flags.filter('one_sided_reviewer').count():,}")

print('\n=== 4. SUSPICION SCORE ===')
print(f'suspicious reviews: {n_flagged:,} of {n_total:,} ({100 * n_flagged / n_total:.2f}%)')
print(prevalence.to_string(index=False))
print(buckets.to_string(index=False))

print('\n=== 5. RATING IMPACT ===')
print(f'products with 10+ reviews: {len(impact):,}   changed by 0.1+ stars: {len(changed):,}   '
      f"inflated: {(impact['change'] <= -0.1).sum():,}   deflated: {(impact['change'] >= 0.1).sum():,}")
print(impact.reindex(impact['change'].abs().sort_values(ascending=False).index)
            .head(12)[['product_title', 'reviews', 'flagged', 'avg_all', 'avg_without_flagged', 'change']]
            .round(2).to_string(index=False, max_colwidth=50))

print('\n=== 6. REVIEWERS ===')
print(f'reviewers with 2+ suspicious reviews: {len(suspicious_reviewers):,}')

## Done - what you have now

| File (in Drive) | What |
|---|---|
| `results/fake_reviews/signal_prevalence_1pct.csv` | how common each signal is |
| `results/fake_reviews/score_distribution_1pct.csv` | suspicion score spread |
| `results/fake_reviews/kmeans_*` | silhouette scores and cluster profiles |
| `results/fake_reviews/product_rating_impact_1pct.csv` | rating before / after removing suspicious reviews |
| `results/fake_reviews/suspicious_reviewers_1pct.csv` | reviewers with 2+ suspicious reviews |
| `results/fake_reviews/manual_check_sheet.csv` | **to label by hand** |
| `results/figures/4b_*.png` | 4 charts |
| `parquet/fake_scores_1pct/` | all signals + score for every review |

**Send back:** the near-duplicate pair count, the suspicious % (section 4), the prevalence table,
the cluster profiles, and the product-impact summary lines (section 5).

**Next:** Phase 4c - recommendation with ALS.